# 05 — Uncertainty & Sensitivity Analysis

Monte Carlo simulation with ±10% weight perturbation across 1,000 iterations.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from utils import load_config, setup_logger
from uncertainty import monte_carlo_uncertainty

logger = setup_logger("05_uncertainty")
hc_cfg = load_config(ROOT / "config/healthcare_config.yml")
ag_cfg = load_config(ROOT / "config/agriculture_config.yml")
PROC   = ROOT / "data/processed"
OUT_R  = ROOT / "outputs/rasters"
OUT_M  = ROOT / "outputs/maps"


## Healthcare — Monte Carlo

In [ ]:
hc_std = {name: PROC / f"hc_{name}_std.tif"
           for name in hc_cfg["criteria"]}
hc_weights = {name: cfg["weight"] for name, cfg in hc_cfg["criteria"].items()}
hc_mask = OUT_R / "healthcare_constraint_mask.tif"

available = {k: v for k, v in hc_std.items() if v.exists()}
if len(available) == len(hc_std):
    unc_hc = monte_carlo_uncertainty(
        criteria_raster_paths=available,
        base_weights=hc_weights,
        output_dir=OUT_R,
        prefix="healthcare",
        n_iterations=hc_cfg["uncertainty"]["monte_carlo_iterations"],
        perturbation_pct=hc_cfg["uncertainty"]["weight_perturbation_pct"],
        constraint_mask_path=hc_mask if hc_mask.exists() else None,
        seed=hc_cfg["uncertainty"]["seed"],
    )
    print("\nHealthcare uncertainty outputs:")
    for k, p in unc_hc.items():
        print(f"  {k:<10} -> {p}")
else:
    print(f"Only {len(available)}/{len(hc_std)} rasters ready. Run notebook 03 first.")
    unc_hc = {}


## Agriculture — Monte Carlo

In [ ]:
ag_std = {name: PROC / f"ag_{name}_std.tif"
           for name in ag_cfg["criteria"]}
ag_weights = {name: cfg["weight"] for name, cfg in ag_cfg["criteria"].items()}
ag_mask = OUT_R / "agriculture_constraint_mask.tif"

available_ag = {k: v for k, v in ag_std.items() if v.exists()}
if len(available_ag) == len(ag_std):
    unc_ag = monte_carlo_uncertainty(
        criteria_raster_paths=available_ag,
        base_weights=ag_weights,
        output_dir=OUT_R,
        prefix="agriculture",
        n_iterations=ag_cfg["uncertainty"]["monte_carlo_iterations"],
        perturbation_pct=ag_cfg["uncertainty"]["weight_perturbation_pct"],
        constraint_mask_path=ag_mask if ag_mask.exists() else None,
        seed=ag_cfg["uncertainty"]["seed"],
    )
else:
    print(f"Only {len(available_ag)}/{len(ag_std)} rasters ready. Run notebook 04 first.")
    unc_ag = {}


## Visualise Uncertainty

In [ ]:
def plot_uncertainty(mean_path, std_path, cv_path, title, cmap_suit="RdYlGn"):
    if not (mean_path.exists() and std_path.exists() and cv_path.exists()):
        print(f"Uncertainty rasters not found for: {title}")
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, path, label, cmap in [
        (axes[0], mean_path, "Mean suitability",         cmap_suit),
        (axes[1], std_path,  "Std deviation (σ)",        "Oranges"),
        (axes[2], cv_path,   "Coeff. of variation (CV)", "Reds"),
    ]:
        with rasterio.open(path) as src:
            arr = src.read(1)
            nodata = src.nodata
        arr_m = np.where(arr == nodata, np.nan, arr) if nodata else arr
        im = ax.imshow(arr_m, cmap=cmap, vmin=0)
        ax.set_title(label, fontsize=10)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.03)
    plt.suptitle(f"Uncertainty Analysis — {title}", fontsize=12)
    plt.tight_layout()
    safe = title.lower().replace(" ", "_")
    fig.savefig(OUT_M / f"{safe}_uncertainty_map.png", dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_M / f'{safe}_uncertainty_map.png'}")

plot_uncertainty(
    OUT_R / "healthcare_uncertainty_mean.tif",
    OUT_R / "healthcare_uncertainty_std.tif",
    OUT_R / "healthcare_uncertainty_cv.tif",
    "Healthcare Suitability"
)

plot_uncertainty(
    OUT_R / "agriculture_uncertainty_mean.tif",
    OUT_R / "agriculture_uncertainty_std.tif",
    OUT_R / "agriculture_uncertainty_cv.tif",
    "Agriculture Suitability",
    cmap_suit="YlGn"
)


## Sensitivity Analysis — Single Criterion Removal

In [ ]:
from suitability_model import weighted_linear_combination
import pandas as pd

def sensitivity_single_removal(std_rasters, weights, mask_path, label):
    """Remove one criterion at a time and measure rank correlation vs base model."""
    from scipy.stats import spearmanr
    base_path = OUT_R / f"{label}_suitability.tif"
    if not base_path.exists():
        print(f"Base model not found for {label}.")
        return

    with rasterio.open(base_path) as src:
        base = src.read(1).flatten()
    valid = ~np.isnan(base)

    results = []
    for drop_name in weights:
        reduced = {k: v for k, v in std_rasters.items() if k != drop_name}
        reduced_weights = {k: v for k, v in weights.items() if k != drop_name}
        total = sum(reduced_weights.values())
        reduced_weights = {k: v/total for k, v in reduced_weights.items()}
        tmp_path = OUT_R / f"_tmp_{label}_{drop_name}.tif"
        available = {k: v for k, v in reduced.items() if v.exists()}
        if not available:
            continue
        weighted_linear_combination(available, reduced_weights, tmp_path,
                                    mask_path if mask_path.exists() else None)
        with rasterio.open(tmp_path) as src:
            alt = src.read(1).flatten()
        both_valid = valid & ~np.isnan(alt)
        rho, _ = spearmanr(base[both_valid], alt[both_valid])
        results.append({"removed": drop_name, "spearman_rho": round(rho, 4),
                        "delta_rho": round(1 - rho, 4)})
        tmp_path.unlink(missing_ok=True)

    df = pd.DataFrame(results).sort_values("delta_rho", ascending=False)
    print(f"\nSensitivity — {label.title()} (Spearman ρ vs base model):")
    print(df.to_string(index=False))
    print("\n  Higher delta_rho = more sensitive to removing that criterion.")
    return df

hc_std_loaded = {name: PROC / f"hc_{name}_std.tif" for name in hc_cfg["criteria"]}
hc_weights    = {name: cfg["weight"] for name, cfg in hc_cfg["criteria"].items()}
hc_sens = sensitivity_single_removal(hc_std_loaded, hc_weights,
                                      OUT_R / "healthcare_constraint_mask.tif",
                                      "healthcare")
